In [ ]:
import json
import pandas as pd

from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient, models
from sentence_transformers import SentenceTransformer

In [5]:
COLLECTION_NAME = "ms_marco_hybrid"

client = QdrantClient(
    url="http://localhost:6333"
)

# Dense
dense_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

# Sparse BM25
sparse_model = SparseTextEmbedding(
    model_name="Qdrant/bm25"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3309.20it/s]


# Dens Search

In [6]:
# ============================================================
# 1. Dense Semantic Search
# ============================================================

def dense_search(
    query: str,
    limit: int = 5
):
    """
    Perform semantic search using dense embeddings.

    Args:
        query (str): Search query provided by the user.
        limit (int): Maximum number of results to return.

    Returns:
        List of Qdrant search results.
    """

    # --------------------------------------------------------
    # Generate Dense Embedding
    # --------------------------------------------------------
    # Convert the query into a dense vector using the same
    # SentenceTransformer model used during indexing.
    #
    # The embedding is normalized so that cosine similarity
    # can be used effectively for semantic search.
    # --------------------------------------------------------

    dense_vector = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()


    # --------------------------------------------------------
    # Search Qdrant
    # --------------------------------------------------------
    # Search only within the "dense" vector space.
    #
    # with_payload=True ensures that the stored metadata,
    # such as text, query, URL, and chunk_id, is returned.
    # --------------------------------------------------------

    results = client.query_points(
        collection_name=COLLECTION_NAME,
        query=dense_vector,
        using="dense",
        limit=limit,
        with_payload=True
    )


    # Return only the individual search points
    return results.points

In [7]:
# ============================================================
# 2. Run Semantic Search
# ============================================================

query = "how much do bartenders make"


# Retrieve the top 10 semantically similar passages
semantic_results = dense_search(
    query=query,
    limit=10
)


# ============================================================
# 3. Display Search Results
# ============================================================

print("== Semantic Search Results ==\n")

# Print table header
print(f"{'Rank':<6}{'Score':<12}{'Chunk ID'}")

print("-" * 35)


# Display each result with its rank, similarity score,
# and corresponding chunk ID.
for rank, result in enumerate(semantic_results, start=1):

    print(
        f"{rank:<6}"
        f"{result.score:<12.4f}"
        f"{result.payload['chunk_id']}"
    )

== Semantic Search Results ==

Rank  Score       Chunk ID
-----------------------------------
1     0.8828      9653_2
2     0.8376      9653_6
3     0.8204      9653_5
4     0.8059      9653_4
5     0.8026      9653_1
6     0.7671      9653_7
7     0.7475      9653_0
8     0.7249      9653_3
9     0.5774      11243_1
10    0.5052      11243_0


# Hybrid Search

In [8]:
# ============================================================
# Hybrid Search: Dense + Sparse BM25 Retrieval
# ============================================================

def hybrid_search(
    query: str,
    limit: int = 5,
    candidate_limit: int = 20
):
    """
    Perform hybrid search using dense semantic embeddings
    and sparse BM25 embeddings.

    The search process consists of:
        1. Generate a dense embedding for the query.
        2. Generate a sparse BM25 embedding for the query.
        3. Retrieve candidates independently from both
           dense and sparse vector spaces.
        4. Combine the results using Reciprocal Rank Fusion (RRF).
        5. Return the final top-ranked results.

    Args:
        query (str):
            The search query.

        limit (int):
            Number of final results to return.

        candidate_limit (int):
            Number of candidate results retrieved from each
            search method before applying RRF fusion.

    Returns:
        List of Qdrant search results.
    """

    # ========================================================
    # 1. Generate Dense Query Embedding
    # ========================================================
    # Convert the query into a dense semantic vector using
    # the same SentenceTransformer model used during indexing.
    #
    # Normalization allows the dense vector to work correctly
    # with cosine similarity.
    # ========================================================

    dense_vector = dense_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()


    # ========================================================
    # 2. Generate Sparse BM25 Query Embedding
    # ========================================================
    # Convert the query into a sparse BM25 representation.
    #
    # BM25 provides lexical matching, which is particularly
    # useful when exact keywords or terms are important.
    # ========================================================

    sparse_vector = list(
        sparse_model.embed([query])
    )[0]


    # Convert the sparse embedding into Qdrant's
    # SparseVector format.
    sparse_query = models.SparseVector(
        indices=sparse_vector.indices.tolist(),
        values=sparse_vector.values.tolist()
    )


    # ========================================================
    # 3. Perform Hybrid Search in Qdrant
    # ========================================================
    # Qdrant performs two independent candidate searches:
    #
    #   - Dense search  -> semantic similarity
    #   - Sparse search -> BM25 lexical matching
    #
    # The candidate results are then combined using
    # Reciprocal Rank Fusion (RRF).
    # ========================================================

    results = client.query_points(
        collection_name=COLLECTION_NAME,

        # ----------------------------------------------------
        # Retrieve candidate results from both vector spaces
        # ----------------------------------------------------

        prefetch=[
            # Dense semantic candidates
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=candidate_limit
            ),

            # Sparse BM25 candidates
            models.Prefetch(
                query=sparse_query,
                using="sparse",
                limit=candidate_limit
            )
        ],

        # ----------------------------------------------------
        # Combine dense and sparse rankings using RRF
        # ----------------------------------------------------

        query=models.FusionQuery(
            fusion=models.Fusion.RRF
        ),

        # Number of final results returned after fusion
        limit=limit,

        # Include stored metadata/payload in the results
        with_payload=True
    )


    # ========================================================
    # 4. Return Final Hybrid Results
    # ========================================================

    return results.points

In [9]:
# ============================================================
# Run Hybrid Search
# ============================================================

# Define the search query
query = "how much do bartenders make"


# Retrieve the top 10 results using hybrid search.
#
# candidate_limit specifies how many candidates are retrieved
# from each retrieval method (dense and sparse) before the
# results are combined using Reciprocal Rank Fusion (RRF).
hybrid_results = hybrid_search(
    query=query,
    limit=10,
    candidate_limit=20
)


# ============================================================
# Display Hybrid Search Results
# ============================================================

# Print the table header
print(f"{'Rank':<6}{'Score':<12}{'Chunk ID'}")

print("-" * 35)


# Display each result with:
#   - Rank
#   - RRF score
#   - Chunk ID
for rank, result in enumerate(hybrid_results, start=1):

    print(
        f"{rank:<6}"
        f"{result.score:<12.4f}"
        f"{result.payload['chunk_id']}"
    )

Rank  Score       Chunk ID
-----------------------------------
1     0.6111      9653_3
2     0.6111      9653_2
3     0.5833      9653_5
4     0.5000      9653_6
5     0.4500      9653_4
6     0.3429      9653_7
7     0.2917      9653_1
8     0.2679      9653_0
9     0.1909      11243_0
10    0.1769      11243_1


# Evaluate

In [10]:
# ============================================================
# 2. Precision@K
# ============================================================

def precision_at_k(
    retrieved_ids,
    expected_ids,
    k
):
    """
    Calculate Precision@K.

    Precision@K measures the proportion of retrieved documents
    in the top K results that are relevant.

    Formula:
        Precision@K = Relevant Retrieved Documents / K

    Args:
        retrieved_ids: Chunk IDs returned by the retriever.
        expected_ids: Ground-truth relevant chunk IDs.
        k: Number of top results to evaluate.

    Returns:
        Precision score between 0 and 1.
    """

    # Consider only the first K retrieved documents
    retrieved_ids = retrieved_ids[:k]

    # Convert expected IDs to a set for efficient lookup
    expected_ids = set(expected_ids)


    # Count how many retrieved chunks are relevant
    relevant = sum(
        chunk_id in expected_ids
        for chunk_id in retrieved_ids
    )


    # Calculate Precision@K
    return relevant / k


# ============================================================
# 3. Recall@K
# ============================================================

def recall_at_k(
    retrieved_ids,
    expected_ids,
    k
):
    """
    Calculate Recall@K.

    Recall@K measures how many of the relevant documents
    were successfully retrieved within the top K results.

    Formula:
        Recall@K = Relevant Retrieved Documents /
                   Total Relevant Documents

    Args:
        retrieved_ids: Chunk IDs returned by the retriever.
        expected_ids: Ground-truth relevant chunk IDs.
        k: Number of top results to evaluate.

    Returns:
        Recall score between 0 and 1.
    """

    # Consider only the first K retrieved documents and
    # convert them to a set to avoid duplicate IDs.
    retrieved_ids = set(retrieved_ids[:k])

    # Convert ground-truth IDs to a set
    expected_ids = set(expected_ids)


    # Find the intersection between retrieved and expected IDs
    relevant = len(
        retrieved_ids & expected_ids
    )


    # Calculate Recall@K
    return relevant / len(expected_ids)

In [11]:
# ============================================================
# 4. Evaluate Retriever
# ============================================================

def evaluate_retriever(
    search_function,
    golden_data,
    top_k=(5, 10),
    name="dense"
):
    """
    Evaluate a retrieval function using Precision@K and Recall@K.

    Each query is searched once using the maximum K value.
    The returned results are then evaluated at each requested K.

    Args:
        search_function:
            Retrieval function to evaluate.
            Example: dense_search or hybrid_search.

        golden_data:
            Ground-truth dataset containing queries and their
            expected relevant chunk IDs.

        top_k:
            Tuple containing the K values to evaluate.

        name:
            Name of the retriever, used in the output DataFrame.

    Returns:
        pandas.DataFrame containing query-level evaluation results.
    """

    results = []

    # Determine the maximum K so that we only need to perform
    # one search per query.
    max_k = max(top_k)


    try:

        # ----------------------------------------------------
        # Evaluate every query in the golden dataset
        # ----------------------------------------------------

        for item in golden_data:

            # Extract query and ground-truth relevant chunks
            query = item["query"]
            expected_ids = item["chunk_ids"]


            # ------------------------------------------------
            # Run the retriever once using the maximum K
            # ------------------------------------------------

            search_results = search_function(
                query,
                limit=max_k
            )


            # Extract chunk IDs from the retrieved results
            retrieved_ids = [
                result.payload["chunk_id"]
                for result in search_results
            ]


            # ------------------------------------------------
            # Calculate metrics for each K
            # ------------------------------------------------

            for k in top_k:

                results.append({

                    # Query information
                    "query_id": item["query_ids"],
                    "query": query,

                    # Retriever information
                    "retriever": name,

                    # Evaluation cutoff
                    "k": k,

                    # Precision@K
                    "precision": precision_at_k(
                        retrieved_ids,
                        expected_ids,
                        k
                    ),

                    # Recall@K
                    "recall": recall_at_k(
                        retrieved_ids,
                        expected_ids,
                        k
                    )
                })


    # --------------------------------------------------------
    # Handle unexpected errors during evaluation
    # --------------------------------------------------------

    except Exception as e:

        print(f"Evaluation failed: {e}")

        import traceback
        traceback.print_exc()


    # Convert the evaluation results into a DataFrame
    return pd.DataFrame(results)


In [12]:
# Path to the evaluation dataset
GOLDEN_DATA_FILE = "data/golden_dataset_question_answer.json"


# Load the JSON evaluation dataset
with open(GOLDEN_DATA_FILE, "r") as f:
    golden_data = json.load(f)["documents"]


print(
    f"Loaded {len(golden_data)} evaluation queries."
)

Loaded 40 evaluation queries.


In [13]:
# ============================================================
# 6. Evaluate Dense Retriever
# ============================================================

dense_results = evaluate_retriever(
    search_function=dense_search,
    golden_data=golden_data,
    top_k=(5, 10),
    name="Dense"
)


# ============================================================
# 7. Evaluate Hybrid Retriever
# ============================================================

hybrid_results = evaluate_retriever(
    search_function=hybrid_search,
    golden_data=golden_data,
    top_k=(5, 10),
    name="Hybrid"
)


In [14]:

# ============================================================
# 8. Combine Evaluation Results
# ============================================================

# Combine dense and hybrid evaluation results into
# a single DataFrame for comparison.
all_results = pd.concat(
    [dense_results, hybrid_results],
    ignore_index=True
)


# ============================================================
# 9. Calculate Aggregate Metrics
# ============================================================

# Calculate the average Precision and Recall for each
# retriever at each K value.
summary = (
    all_results
    .groupby(["retriever", "k"])[
        ["precision", "recall"]
    ]
    .mean()
    .reset_index()
)


In [15]:
# ============================================================
# 10. Display Evaluation Summary
# ============================================================

print(summary)

  retriever   k  precision    recall
0     Dense   5     0.1850  0.664167
1     Dense  10     0.1275  0.770000
2    Hybrid   5     0.2000  0.735000
3    Hybrid  10     0.1475  0.951250
